In [1]:
# Cell 1 — Config (13_44)
from pathlib import Path
import pandas as pd, json, re, cv2
from collections import Counter, defaultdict

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MANIFEST = ROOT / "data" / "ocr_frames_13_44" / "manifest.csv"
OUT      = ROOT / "data" / "ocr_13_44"
OUT.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(MANIFEST)
CAM_WEIGHT = {"back_right": 3, "front": 2, "back_left": 1}

print("crops:", len(manifest))
print("per camera:", manifest.groupby("camera").size().to_dict())
print("per colour:", manifest.groupby("color").size().to_dict())
print("bottles:", manifest.groupby(["video","Sequence"]).size().shape[0])
manifest.head()

crops: 394
per camera: {'back_left': 136, 'back_right': 144, 'front': 114}
per colour: {'black': 342, 'cyan': 52}
bottles: 147


,video,Sequence,t_sec,camera,priority,crop_path,full_crop_path,gt_tare,gt_year,color
0,13_44,541904,25.0,back_right,0,data/ocr_frames_13_44/crop_13_44_541904_back_r...,data/ocr_frames_13_44/full_13_44_541904_back_r...,10.66,2039.0,black
1,13_44,541905,32.0,front,1,data/ocr_frames_13_44/crop_13_44_541905_front.jpg,data/ocr_frames_13_44/full_13_44_541905_front.jpg,15.40,2035.0,black
2,13_44,541905,32.0,back_left,2,data/ocr_frames_13_44/crop_13_44_541905_back_l...,data/ocr_frames_13_44/full_13_44_541905_back_l...,15.40,2035.0,black
3,13_44,541905,32.0,back_right,0,data/ocr_frames_13_44/crop_13_44_541905_back_r...,data/ocr_frames_13_44/full_13_44_541905_back_r...,15.40,2035.0,black
4,13_44,541906,39.0,back_left,2,data/ocr_frames_13_44/crop_13_44_541906_back_l...,data/ocr_frames_13_44/full_13_44_541906_back_l...,NaN,NaN,cyan


In [2]:
# Cell 2 — preprocess (kept mild; metal text dislikes heavy blur)
def preprocess(img_bgr):
    g = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    g = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8)).apply(g)
    g = cv2.bilateralFilter(g, 9, 75, 75)
    g = cv2.resize(g, None, fx=3, fy=3, interpolation=cv2.INTER_LANCZOS4)
    return g

In [3]:
import time, cv2, easyocr

def cap_size(img, target_long=640):
    h, w = img.shape[:2]
    L = max(h, w)
    if L <= target_long:
        return img
    s = target_long / L
    return cv2.resize(img, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)

reader = easyocr.Reader(['en'], gpu=False, verbose=False)

sample = manifest.sample(8, random_state=0)
for r in sample.itertuples(index=False):
    img = cv2.imread(str(ROOT / r.crop_path))
    h, w = img.shape[:2]

    t0 = time.time()
    reader.readtext(img, detail=0)
    t_orig = time.time() - t0

    img_small = cap_size(img, 640)
    t0 = time.time()
    reader.readtext(img_small, detail=0, canvas_size=736, mag_ratio=1.0)
    t_fast = time.time() - t0

    print(f"{r.camera:10s} {r.Sequence:6d}  size={h}x{w}  orig={t_orig*1000:.0f} ms  fast={t_fast*1000:.0f} ms")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/torch/ao/nn/quantized/dynamic/modules/rnn.py:162: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/quantized/Quantizer.cpp:116.)
  w_ih = torch.quantize_per_tensor(


front      541948  size=753x897  orig=2059 ms  fast=354 ms


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


back_right 542012  size=829x911  orig=2088 ms  fast=737 ms
back_right 541929  size=808x1014  orig=1544 ms  fast=431 ms
back_right 541964  size=830x1226  orig=2280 ms  fast=351 ms
back_left  542030  size=834x938  orig=1690 ms  fast=377 ms
back_right 542044  size=828x927  orig=1441 ms  fast=374 ms
front      541995  size=827x1035  orig=2012 ms  fast=359 ms
back_right 542032  size=830x947  orig=1175 ms  fast=386 ms


In [4]:
import torch, os, time, cv2, easyocr

# 1. use all CPU cores
torch.set_num_threads(os.cpu_count())
torch.set_num_interop_threads(1)
print("torch threads:", torch.get_num_threads(), "| cpu count:", os.cpu_count())

# 2. helper
def cap_size(img, target_long=640):
    h, w = img.shape[:2]
    L = max(h, w)
    if L <= target_long:
        return img
    s = target_long / L
    return cv2.resize(img, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)

# 3. reader
reader = easyocr.Reader(['en'], gpu=False, verbose=False)

# 4. quick benchmark on 5 crops
sample = manifest.sample(5, random_state=0)
for r in sample.itertuples(index=False):
    img = cv2.imread(str(ROOT / r.crop_path))
    img = cap_size(img, 640)
    t0 = time.time()
    reader.readtext(img, detail=0, canvas_size=736, mag_ratio=1.0,
                    batch_size=1, decoder="greedy", paragraph=False)
    print(f"{r.camera:10s} {r.Sequence:>8}  size={img.shape[0]}x{img.shape[1]}  {(time.time()-t0)*1000:.0f} ms")

torch threads: 10 | cpu count: 10
front        541948  size=537x640  535 ms
back_right   542012  size=582x640  682 ms
back_right   541929  size=509x640  426 ms
back_right   541964  size=433x640  286 ms
back_left    542030  size=569x640  365 ms


In [5]:
# Cell 3 — cap_size defined first, then OCR loop
import easyocr, time

def cap_size(img, target_long=640):
    h, w = img.shape[:2]
    L = max(h, w)
    if L <= target_long:
        return img
    s = target_long / L
    return cv2.resize(img, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)

reader = easyocr.Reader(['en'], gpu=False, verbose=False)

TARE_RE = re.compile(r'(\d{1,2})[,.](\d)\s*[Kk][Gg]')
YEAR_RE = re.compile(r'\b(19|20)\d{2}\b')
STICKER_KG = {12.5, 10.5}

def parse(txts):
    joined = " ".join(txts).replace(" ", "")
    tare = None
    m = TARE_RE.search(joined)
    if m:
        v = float(f"{m.group(1)}.{m.group(2)}")
        if v not in STICKER_KG and 5.0 <= v <= 25.0:
            tare = v
        else:
            for tok in re.findall(r'\d{1,2}[,.]\d', joined):
                vv = float(tok.replace(",", "."))
                if vv not in STICKER_KG and 5.0 <= vv <= 25.0:
                    tare = vv; break
    year = None
    for tok in txts:
        y = YEAR_RE.search(tok)
        if y and 1990 <= int(y.group(0)) <= 2035:
            year = int(y.group(0)); break
    return tare, year

manifest = manifest.drop_duplicates(subset=["video", "Sequence", "camera"]).reset_index(drop=True)
print(f"manifest: {len(manifest)} crops")

crop_rows = []
t_start = time.time()

for i, r in enumerate(manifest.itertuples(index=False), 1):
    img = cv2.imread(str(ROOT / r.crop_path))
    if img is None:
        continue
    img = cap_size(img, 640)
    txts = reader.readtext(
        img, detail=0,
        canvas_size=736,
        mag_ratio=1.0,
        batch_size=8,
        decoder="greedy",
    )
    tare, year = parse(txts)
    crop_rows.append({
        "video": r.video, "Sequence": int(r.Sequence),
        "camera": r.camera, "priority": int(r.priority),
        "color": r.color,
        "gt_tare": r.gt_tare, "gt_year": r.gt_year,
        "raw": str(txts),
        "tare": tare, "year": year,
    })
    if i % 40 == 0:
        elapsed = time.time() - t_start
        per_crop = elapsed / i
        remaining = per_crop * (len(manifest) - i)
        print(f"  {i}/{len(manifest)}  ({elapsed:.0f}s elapsed, ~{remaining:.0f}s left, {per_crop*1000:.0f} ms/crop)")

crop_df = pd.DataFrame(crop_rows)
crop_df.to_csv(OUT / "ocr_per_crop.csv", index=False)
print(f"\ncrops OCR'd: {len(crop_df)}",
      f"| crops with tare: {int(crop_df.tare.notna().sum())}",
      f"| crops with year: {int(crop_df.year.notna().sum())}",
      f"| total time: {time.time()-t_start:.0f}s")

manifest: 394 crops
  40/394  (21s elapsed, ~188s left, 532 ms/crop)
  80/394  (48s elapsed, ~190s left, 604 ms/crop)
  120/394  (75s elapsed, ~170s left, 622 ms/crop)
  160/394  (103s elapsed, ~150s left, 643 ms/crop)
  200/394  (133s elapsed, ~129s left, 663 ms/crop)
  240/394  (161s elapsed, ~103s left, 671 ms/crop)
  280/394  (193s elapsed, ~79s left, 691 ms/crop)
  320/394  (227s elapsed, ~52s left, 709 ms/crop)
  360/394  (253s elapsed, ~24s left, 702 ms/crop)

crops OCR'd: 394 | crops with tare: 1 | crops with year: 22 | total time: 276s


In [7]:
# Cell 4 — weighted majority vote per bottle, per-camera and per-colour metrics
def weighted_vote(rows, field):
    scores = defaultdict(float)
    for r in rows:
        v = r.get(field)
        if v is None or pd.isna(v):
            continue
        scores[v] += CAM_WEIGHT.get(r.get("camera"), 1)
    if not scores:
        return None
    return max(scores.items(), key=lambda kv: kv[1])[0]

def plain_vote(rows, field):
    vals = [r[field] for r in rows if r.get(field) is not None and not pd.isna(r.get(field))]
    return Counter(vals).most_common(1)[0][0] if vals else None

per_bottle = []
for (vk, seq), grp in crop_df.groupby(["video", "Sequence"]):
    rows = grp.to_dict("records")
    per_bottle.append({
        "video": vk, "Sequence": seq,
        "color": rows[0]["color"],
        "n_cameras_attempted": len(rows),
        "cameras": "+".join(sorted(r["camera"] for r in rows)),
        "tare":       weighted_vote(rows, "tare"),
        "year":       weighted_vote(rows, "year"),
        "tare_plain": plain_vote(rows, "tare"),
        "year_plain": plain_vote(rows, "year"),
        "gt_tare":    rows[0]["gt_tare"],
        "gt_year":    rows[0]["gt_year"],
    })

bottle_df = pd.DataFrame(per_bottle)
bottle_df["tare_ok"]       = (bottle_df.tare       == bottle_df.gt_tare).astype(int)
bottle_df["year_ok"]       = (bottle_df.year       == bottle_df.gt_year).astype(int)
bottle_df["tare_ok_plain"] = (bottle_df.tare_plain == bottle_df.gt_tare).astype(int)
bottle_df["year_ok_plain"] = (bottle_df.year_plain == bottle_df.gt_year).astype(int)

# overall
n = len(bottle_df)
overall = {
    "n_bottles":               n,
    "tare_exact_match_weighted": round(int(bottle_df.tare_ok.sum()) / n, 3),
    "year_exact_match_weighted": round(int(bottle_df.year_ok.sum()) / n, 3),
    "both_exact_match_weighted": round(int(((bottle_df.tare_ok == 1) & (bottle_df.year_ok == 1)).sum()) / n, 3),
    "tare_exact_match_plain":    round(int(bottle_df.tare_ok_plain.sum()) / n, 3),
    "year_exact_match_plain":    round(int(bottle_df.year_ok_plain.sum()) / n, 3),
}

# per-colour
per_color = {}
for color, grp in bottle_df.groupby("color"):
    m = len(grp)
    per_color[color] = {
        "bottles": m,
        "tare_exact_match_weighted": round(int(grp.tare_ok.sum()) / m, 3),
        "year_exact_match_weighted": round(int(grp.year_ok.sum()) / m, 3),
        "both_exact_match_weighted": round(int(((grp.tare_ok == 1) & (grp.year_ok == 1)).sum()) / m, 3),
    }

# back-right alone
br = crop_df[crop_df.camera == "back_right"].copy()
br = br.drop_duplicates(subset=["video", "Sequence"]).reset_index(drop=True)

# diagnostics: guard against every type of silent surprise
print("br rows:", len(br))
print("br columns:", list(br.columns))

br_metrics = {
    "back_right_crops":         int(len(br)),
    "back_right_tare_hits":     int(br["tare"].notna().sum()),
    "back_right_year_hits":     int(br["year"].notna().sum()),
    "back_right_tare_correct":  int((br["tare"] == br["gt_tare"]).sum()),
    "back_right_year_correct":  int((br["year"] == br["gt_year"]).sum()),
}
print("br_metrics:", br_metrics)

result = {
    "overall": overall,
    "per_color": per_color,
    "back_right_only": br_metrics,
    "coverage": {
        "gt_bottles_targeted": int(len(manifest.groupby(['video','Sequence']).size())),
        "bottles_with_3_cameras": int((bottle_df.n_cameras_attempted == 3).sum()),
        "bottles_with_2_cameras": int((bottle_df.n_cameras_attempted == 2).sum()),
        "bottles_with_1_camera":  int((bottle_df.n_cameras_attempted == 1).sum()),
        "total_crops":            int(len(crop_df)),
    },
}
print(json.dumps(result, indent=2))
(OUT / "ocr_metrics.json").write_text(json.dumps(result, indent=2))
bottle_df.to_csv(OUT / "ocr_per_bottle.csv", index=False)
print("\nsaved:", (OUT / "ocr_metrics.json").relative_to(ROOT))

br rows: 144
br columns: ['video', 'Sequence', 'camera', 'priority', 'color', 'gt_tare', 'gt_year', 'raw', 'tare', 'year']
br_metrics: {'back_right_crops': 144, 'back_right_tare_hits': 0, 'back_right_year_hits': 5, 'back_right_tare_correct': 0, 'back_right_year_correct': 4}
{
  "overall": {
    "n_bottles": 147,
    "tare_exact_match_weighted": 0.0,
    "year_exact_match_weighted": 0.034,
    "both_exact_match_weighted": 0.0,
    "tare_exact_match_plain": 0.0,
    "year_exact_match_plain": 0.034
  },
  "per_color": {
    "black": {
      "bottles": 128,
      "tare_exact_match_weighted": 0.0,
      "year_exact_match_weighted": 0.039,
      "both_exact_match_weighted": 0.0
    },
    "cyan": {
      "bottles": 19,
      "tare_exact_match_weighted": 0.0,
      "year_exact_match_weighted": 0.0,
      "both_exact_match_weighted": 0.0
    }
  },
  "back_right_only": {
    "back_right_crops": 144,
    "back_right_tare_hits": 0,
    "back_right_year_hits": 5,
    "back_right_tare_correct": 0,

In [9]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

gram_map = {}
for vk, path, sheet in [
    ("13_44", ROOT/"data"/"log"/"13_44_ground_truth_ai_app.xlsx", "13_44_ground_truth"),
]:
    df = pd.read_excel(path, sheet_name=sheet)
    if "Sequence" not in df.columns:
        df["Sequence"] = range(len(df))

    # skip note rows (text in Sequence) and rows without Pr_tare_gram
    df["Sequence"] = pd.to_numeric(df["Sequence"], errors="coerce")
    df["Pr_tare_gram"] = pd.to_numeric(df["Pr_tare_gram"], errors="coerce")
    df = df.dropna(subset=["Sequence", "Pr_tare_gram"]).copy()
    df["Sequence"] = df["Sequence"].astype(int)

    for _, r in df.iterrows():
        gram_map[(vk, int(r["Sequence"]))] = float(r["Pr_tare_gram"]) / 1000.0

print("gram_map entries:", len(gram_map))

# attach gram_ref to bottle_df and compute plausibility
bottle_df["gram_ref"] = bottle_df.apply(
    lambda r: gram_map.get((r["video"], int(r["Sequence"])), None), axis=1
)

bottle_df["tare_plausible"] = bottle_df.apply(
    lambda r: pd.notna(r["tare"]) and pd.notna(r["gram_ref"])
              and abs(float(r["tare"]) - float(r["gram_ref"])) <= 0.5,
    axis=1,
)

n_total = len(bottle_df)
n_plausible = int(bottle_df.tare_plausible.sum())

print(f"\nbottles: {n_total}")
print(f"bottles with a plausible tare (|OCR - Pr_tare_gram| <= 0.5 kg): {n_plausible}")
print("\nthe plausible ones:")
if n_plausible > 0:
    print(bottle_df[bottle_df.tare_plausible][
        ["video", "Sequence", "tare", "gram_ref", "gt_tare"]
    ].to_string(index=False))
else:
    print("  (none)")

gram_map entries: 143

bottles: 147
bottles with a plausible tare (|OCR - Pr_tare_gram| <= 0.5 kg): 1

the plausible ones:
video  Sequence  tare  gram_ref  gt_tare
13_44    541923  10.4      10.5     10.5


In [10]:
# look at the 10 bottles closest to the ground truth (most plausible)
bottle_df["gap"] = (bottle_df["tare"] - bottle_df["gram_ref"]).abs()
print(bottle_df.nsmallest(10, "gap")[["video", "Sequence", "tare", "gram_ref", "gt_tare", "gap"]].to_string(index=False))

video  Sequence  tare  gram_ref  gt_tare  gap
13_44    541923  10.4      10.5    10.50  0.1
13_44    541904   NaN      10.7    10.66  NaN
13_44    541905   NaN      15.4    15.40  NaN
13_44    541906   NaN      10.8      NaN  NaN
13_44    541907   NaN      16.4    16.40  NaN
13_44    541908   NaN      10.5    10.50  NaN
13_44    541909   NaN      10.8    10.80  NaN
13_44    541910   NaN      15.4      NaN  NaN
13_44    541911   NaN      10.8      NaN  NaN
13_44    541912   NaN      10.9    10.90  NaN


In [12]:
# Cell 5 — sanity check
print("crops:", len(crop_df),
      "| with tare:", int(crop_df.tare.notna().sum()),
      "| with year:", int(crop_df.year.notna().sum()))
print("\n5 random raw-text examples:")
for t in crop_df.sample(min(5, len(crop_df)), random_state=0)["raw"].tolist():
    print("  ", t[:140])
print("\n5 random crops where tare WAS parsed:")
parsed = crop_df[crop_df.tare.notna()]
if len(parsed) > 0:
    for _, r in parsed.sample(min(5, len(parsed)), random_state=0).iterrows():
        print(f"  {r.video} seq={r.Sequence} cam={r.camera} tare={r.tare} gt={r.gt_tare}  raw={r.raw[:80]}")
else:
    print("  (none)")

crops: 394 | with tare: 1 | with year: 22

5 random raw-text examples:
   []
   ['Ut9s', '15,316 -', 'UN9OS', '2035']
   ['RoFet']
   ['"ROPAN']
   ['PRIMAC']

5 random crops where tare WAS parsed:
  13_44 seq=541923 cam=front tare=10.4 gt=10.5  raw=['T Dar', '1R4', '3', "'10,4", 'KG:', "'2033"]


In [13]:
# Diagnostic — no GT matching, just what OCR saw per bottle (13_44, uses crop_df from above)
import pandas as pd

for (vk, seq), grp in crop_df.groupby(["video", "Sequence"]):
    gt_tare = grp.iloc[0]["gt_tare"]
    gt_year = grp.iloc[0]["gt_year"]
    colour  = grp.iloc[0]["color"]

    print(f"\n=== {vk}  Sequence {seq}  ({colour}, GT: tare={gt_tare}  year={gt_year}) ===")
    for _, r in grp.iterrows():
        cam   = r["camera"]
        raw   = str(r["raw"])[:140]
        tare  = r["tare"] if pd.notna(r["tare"]) else "—"
        year  = r["year"] if pd.notna(r["year"]) else "—"
        print(f"  {cam:10s}  tare={str(tare):6s}  year={str(year):4s}  raw={raw}")


=== 13_44  Sequence 541904  (black, GT: tare=10.66  year=2039.0) ===
  back_right  tare=—       year=—     raw=['71885']

=== 13_44  Sequence 541905  (black, GT: tare=15.4  year=2035.0) ===
  front       tare=—       year=—     raw=[]
  back_left   tare=—       year=—     raw=[]
  back_right  tare=—       year=—     raw=['UN', 'Un', '41965', 'lon']

=== 13_44  Sequence 541906  (cyan, GT: tare=nan  year=nan) ===
  back_left   tare=—       year=—     raw=['PRIM']
  back_right  tare=—       year=—     raw=['IAGAZ']

=== 13_44  Sequence 541907  (black, GT: tare=16.4  year=2036.0) ===
  front       tare=—       year=—     raw=['615', 'N', "'.`"]
  back_left   tare=—       year=2033.0  raw=['TG', 'P15Y-T', '2033']
  back_right  tare=—       year=—     raw=['10 .3', 'UNi965', 'KG?', 'Pisy', '<']

=== 13_44  Sequence 541908  (black, GT: tare=10.5  year=2035.0) ===
  front       tare=—       year=—     raw=[]
  back_left   tare=—       year=—     raw=['65', '76', 'Uv', 'Pr', '05', '1955', '44L